# 🤖 Agentic RAG System — Google Colab

**Architecture overview:**
- 📄 **Document Ingestion** — upload your own PDF from local disk
- 🔍 **VectorStore** — FAISS + HuggingFace embeddings (no OpenAI key needed)
- 🧠 **LLM** — Groq (`llama-3.3-70b-versatile`) via `langchain-groq`
- 🔗 **LangGraph** — `retriever node → generate_answer node (direct LLM call)` → answer

> **Before running:** add your `GROQ_API_KEY` in *Colab Secrets* (🔑 left sidebar → Secrets).

## 1 · Install dependencies

In [1]:
# Install all required packages
!pip install -q \
    langchain\
    langchain-community\
    langchain-groq \
    langchain-huggingface \
    langgraph \
    faiss-cpu\
    pydantic \
    pypdf \
    wikipedia \
    sentence-transformers \
    beautifulsoup4 \
    requests

import uuid # Moved import uuid here to ensure it's globally available for early langchain dependencies

print('✅ All packages installed successfully!')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ All packages installed successfully!


In [2]:
# STEP 1: Check versions
import langgraph, langchain_core, langchain_community
# print("langgraph      :", langgraph.__version__) # Removed this line
print("langchain_core :", langchain_core.__version__)
print("langchain_comm :", langchain_community.__version__)

langchain_core : 1.2.19
langchain_comm : 0.4.1


## 2 · Load Groq API Key from Colab Secrets

In [3]:
import os
from google.colab import userdata

# Reads GROQ_API_KEY from Colab Secrets (left sidebar → 🔑 Secrets)
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

print('✅ GROQ_API_KEY loaded!')

✅ GROQ_API_KEY loaded!


## 3 · Configuration

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# src/config/config.py  (inlined for Colab)
# ─────────────────────────────────────────────────────────────────────────────
import os
from langchain_groq import ChatGroq

class Config:
    """Configuration for the Agentic RAG system."""

    # ── Model ────────────────────────────────────────────────────────────────
    LLM_MODEL        = 'llama-3.3-70b-versatile'   # fast & capable Groq model
    EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'          # local HuggingFace embeddings

    # ── Chunking ─────────────────────────────────────────────────────────────
    CHUNK_SIZE       = 500
    CHUNK_OVERLAP    = 50

    @classmethod
    def get_llm(cls):
        """Return a ChatGroq instance."""
        return ChatGroq(
            model=cls.LLM_MODEL,
            api_key=os.environ['GROQ_API_KEY'],
            temperature=0,
        )

print(f'✅ Config ready  |  LLM: {Config.LLM_MODEL}  |  Embeddings: {Config.EMBEDDING_MODEL}')

✅ Config ready  |  LLM: llama-3.3-70b-versatile  |  Embeddings: all-MiniLM-L6-v2


## 4 · RAG State  (`src/state/rag_state.py`)

In [5]:
from typing import List
from pydantic import BaseModel, ConfigDict
from langchain_core.documents import Document

class RAGState(BaseModel):
    """Typed state object that flows through the LangGraph pipeline."""
    question      : str
    retrieved_docs: List[Document] = []
    answer        : str = ''

    model_config = ConfigDict(arbitrary_types_allowed=True)

print('✅ RAGState defined')

✅ RAGState defined


## 5 · Document Processor  (`src/document_ingestion/document_processor.py`)

In [6]:
from typing import List, Union
from pathlib import Path
from langchain_community.document_loaders import (
    PyPDFLoader,
    WebBaseLoader,
    TextLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

class DocumentProcessor:
    """Load documents from PDFs / URLs / TXT files and split into chunks."""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size    = chunk_size
        self.chunk_overlap = chunk_overlap
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    # ── loaders ──────────────────────────────────────────────────────────────
    def load_from_pdf(self, file_path: Union[str, Path]) -> List[Document]:
        """Load a single PDF file."""
        loader = PyPDFLoader(str(file_path))
        return loader.load()

    def load_from_url(self, url: str) -> List[Document]:
        """Load a web page."""
        loader = WebBaseLoader(url)
        return loader.load()

    def load_from_txt(self, file_path: Union[str, Path]) -> List[Document]:
        """Load a plain-text file."""
        loader = TextLoader(str(file_path), encoding='utf-8')
        return loader.load()

    # ── router ────────────────────────────────────────────────────────────────
    def load_documents(self, sources: List[str]) -> List[Document]:
        """
        Load from a mixed list of sources:
          • http(s)://...  → WebBaseLoader
          • *.pdf          → PyPDFLoader
          • *.txt          → TextLoader
        """
        docs: List[Document] = []
        for src in sources:
            src = src.strip()
            if src.startswith('http://') or src.startswith('https://'):
                print(f'  🌐 Loading URL: {src}')
                docs.extend(self.load_from_url(src))
            else:
                path = Path(src)
                if path.suffix.lower() == '.pdf':
                    print(f'  📄 Loading PDF: {path.name}')
                    docs.extend(self.load_from_pdf(path))
                elif path.suffix.lower() == '.txt':
                    print(f'  📝 Loading TXT: {path.name}')
                    docs.extend(self.load_from_txt(path))
                else:
                    raise ValueError(f'Unsupported source: {src}  (use URL, .pdf, or .txt)')
        return docs

    # ── splitter ──────────────────────────────────────────────────────────────
    def split_documents(self, documents: List[Document]) -> List[Document]:
        return self.splitter.split_documents(documents)

    # ── combined pipeline ────────────────────────────────────────────────────
    def process(self, sources: List[str]) -> List[Document]:
        """Load + split in one call."""
        raw = self.load_documents(sources)
        chunks = self.split_documents(raw)
        print(f'  ✂️  {len(raw)} pages → {len(chunks)} chunks')
        return chunks

print('✅ DocumentProcessor defined')

✅ DocumentProcessor defined


## 6 · VectorStore  (`src/vectorstore/vectorstore.py`)

In [7]:
from typing import List
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

class VectorStore:
    """FAISS-backed vector store using local HuggingFace sentence-transformer embeddings."""

    def __init__(self, model_name: str = Config.EMBEDDING_MODEL):
        print(f'  🔄 Loading embedding model: {model_name} …')
        self.embedding   = HuggingFaceEmbeddings(model_name=model_name)
        self.vectorstore = None
        self.retriever   = None
        print('  ✅ Embedding model loaded')

    def create_vectorstore(self, documents: List[Document]):
        """Embed documents and build FAISS index."""
        print(f'  🔍 Embedding {len(documents)} chunks …')
        self.vectorstore = FAISS.from_documents(documents, self.embedding)
        self.retriever   = self.vectorstore.as_retriever(search_kwargs={'k': 6})
        print('  ✅ FAISS index created')

    def get_retriever(self):
        if self.retriever is None:
            raise ValueError('VectorStore not initialised — call create_vectorstore() first.')
        return self.retriever

    def retrieve(self, query: str) -> List[Document]:
        return self.get_retriever().invoke(query)

print('✅ VectorStore defined')

✅ VectorStore defined


## 7 · LangGraph Nodes  (`src/node/reactnode.py`)

In [8]:
# src/node/reactnode.py  (inlined for Colab)
# Direct LLM call — no tool-calling needed, fully Groq-compatible
from typing import List, Optional
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

class RAGNodes:
    """Contains node functions for the LangGraph RAG workflow."""

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm       = llm

    # ── Node 1: retrieve_docs ─────────────────────────────────────────────
    def retrieve_docs(self, state: RAGState) -> RAGState:
        """Retrieve relevant chunks from the FAISS vector store."""
        docs = self.retriever.invoke(state.question)
        return RAGState(
            question=state.question,
            retrieved_docs=docs,
        )

    # ── Node 2: generate_answer ───────────────────────────────────────────
    def generate_answer(self, state: RAGState) -> RAGState:
        """
        Build context from retrieved docs and call Groq LLM directly.
        No tool-calling — avoids Groq BadRequestError (tool_use_failed).
        """
        # Build numbered context block from retrieved docs
        if state.retrieved_docs:
            context = "\n\n".join(
                f"[{i}] {doc.page_content}"
                for i, doc in enumerate(state.retrieved_docs, start=1)
            )
        else:
            context = "No relevant documents were found in the vector store."

        messages = [
            SystemMessage(content=(
                "You are a helpful assistant. "
                "Answer the user's question using the provided context. "
                "If the context does not contain enough information, say so clearly "
                "and answer from your general knowledge where appropriate."
            )),
            HumanMessage(content=(
                f"Context from document:\n{context}\n\n"
                f"Question: {state.question}"
            )),
        ]

        response = self.llm.invoke(messages)

        return RAGState(
            question=state.question,
            retrieved_docs=state.retrieved_docs,
            answer=response.content,
        )

print('✅ RAGNodes defined  (src/node/reactnode.py)')

✅ RAGNodes defined  (src/node/reactnode.py)


## 8 · Graph Builder  (`src/graph_builder/graph_builder.py`)

In [9]:
from langgraph.graph import StateGraph, END

class GraphBuilder:
    """Assembles and compiles the LangGraph RAG workflow."""

    def __init__(self, retriever, llm):
        self.nodes = RAGNodes(retriever, llm)
        self.graph = None

    def build(self):
        """Wire up nodes and compile the graph."""
        builder = StateGraph(RAGState)

        builder.add_node('retriever', self.nodes.retrieve_docs)
        builder.add_node('responder', self.nodes.generate_answer)

        builder.set_entry_point('retriever')
        builder.add_edge('retriever', 'responder')
        builder.add_edge('responder', END)

        self.graph = builder.compile()
        print('✅ LangGraph compiled  →  retriever ➜ responder ➜ END')
        return self.graph

    def run(self, question: str) -> dict:
        """Execute the pipeline for a given question."""
        if self.graph is None:
            self.build()
        initial_state = RAGState(question=question)
        return self.graph.invoke(initial_state)

print('✅ GraphBuilder defined')

✅ GraphBuilder defined


## 9 · Upload your PDF from local disk

Run this cell — a file-picker will appear. Select your PDF and wait for the upload to finish.

In [10]:
from google.colab import files
import os

print('📂 Please select your PDF file …')
uploaded = files.upload()   # triggers the Colab file-picker

# Get the uploaded file path
pdf_filename = list(uploaded.keys())[0]
PDF_PATH = os.path.abspath(pdf_filename)

print(f'\n✅ Uploaded: {pdf_filename}  ({len(uploaded[pdf_filename]):,} bytes)')
print(f'   Saved to: {PDF_PATH}')

📂 Please select your PDF file …


Saving attention.pdf to attention.pdf

✅ Uploaded: attention.pdf  (2,215,244 bytes)
   Saved to: /content/attention.pdf


## 10 · Initialise the full RAG pipeline

In [11]:
# ── LLM ──────────────────────────────────────────────────────────────────
print('🚀 Initialising Agentic RAG System …\n')
llm = Config.get_llm()
print(f'✅ LLM ready: {Config.LLM_MODEL} (via Groq)\n')

# ── Document Processor ────────────────────────────────────────────────────
print('📄 Loading & chunking PDF …')
doc_processor = DocumentProcessor(
    chunk_size    = Config.CHUNK_SIZE,
    chunk_overlap = Config.CHUNK_OVERLAP,
)
documents = doc_processor.process([PDF_PATH])
print(f'   Total chunks: {len(documents)}\n')

# ── VectorStore ───────────────────────────────────────────────────────────
print('🔍 Building FAISS vector store …')
vector_store = VectorStore()
vector_store.create_vectorstore(documents)
print()

# ── Graph ─────────────────────────────────────────────────────────────────
print('🔗 Building LangGraph …')
graph_builder = GraphBuilder(
    retriever = vector_store.get_retriever(),
    llm       = llm,
)
graph_builder.build()

print('\n🎉 System fully initialised — ready to answer questions!')

🚀 Initialising Agentic RAG System …

✅ LLM ready: llama-3.3-70b-versatile (via Groq)

📄 Loading & chunking PDF …
  📄 Loading PDF: attention.pdf
  ✂️  15 pages → 93 chunks
   Total chunks: 93

🔍 Building FAISS vector store …
  🔄 Loading embedding model: all-MiniLM-L6-v2 …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✅ Embedding model loaded
  🔍 Embedding 93 chunks …
  ✅ FAISS index created

🔗 Building LangGraph …
✅ LangGraph compiled  →  retriever ➜ responder ➜ END

🎉 System fully initialised — ready to answer questions!


## 11 · Ask a single question

In [12]:
import time

question = 'What is LLM?'   # ← change this

print(f'❓ Question: {question}\n')
t0 = time.time()

result = graph_builder.run(question)

elapsed = time.time() - t0
print(f'\n💡 Answer:\n{result["answer"]}')
print(f'\n⏱️  {elapsed:.2f}s  |  {len(result["retrieved_docs"])} docs retrieved')

❓ Question: What is LLM?


💡 Answer:
The provided context does not contain enough information to determine what LLM stands for. However, based on my general knowledge, LLM can stand for Large Language Model, which is a type of artificial intelligence model designed to process and understand human language. But without more context, it's difficult to say for certain what LLM refers to in this specific case.

⏱️  0.77s  |  6 docs retrieved


## 12 · Inspect retrieved source passages

In [13]:
for i, doc in enumerate(result['retrieved_docs'], start=1):
    meta   = doc.metadata
    source = meta.get('source', meta.get('file_path', 'unknown'))
    page   = meta.get('page', '?')
    print(f'--- Doc {i} | source: {source} | page: {page} ---')
    print(doc.page_content[:400])
    print()

--- Doc 1 | source: /content/attention.pdf | page: 0 ---
tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and
efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating
our research.
†Work performed while at Google Brain.
‡Work pe

--- Doc 2 | source: /content/attention.pdf | page: 10 ---
2017.
[19] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks.
In International Conference on Learning Representations , 2017.
[20] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015.
[21] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint
arXiv:1703.10722, 2017.
[22] Zhouhan Lin, Mi

--- Doc 3 | source: /content/attention.pdf | page: 9 ---
arXiv:1607.06450, 2016.


## 13 · Interactive Q&A loop

Type questions one at a time. Enter `quit` or `exit` to stop.

In [14]:
import time

print('💬 Interactive Mode  —  type "quit" to stop\n')
print('─' * 60)

while True:
    try:
        q = input('\nYour question: ').strip()
    except EOFError:
        break

    if not q:
        continue
    if q.lower() in ('quit', 'exit', 'q'):
        print('👋 Goodbye!')
        break

    t0     = time.time()
    result = graph_builder.run(q)
    print(f'\n💡 Answer:\n{result["answer"]}')
    print(f'\n⏱️  {time.time()-t0:.2f}s  |  {len(result["retrieved_docs"])} docs retrieved')
    print('─' * 60)

💬 Interactive Mode  —  type "quit" to stop

────────────────────────────────────────────────────────────

Your question: what is llm?

💡 Answer:
The provided context does not contain enough information to determine what "LLM" stands for. However, based on my general knowledge, I can suggest that "LLM" could potentially refer to "Large Language Model," which is a type of artificial intelligence model designed to process and understand human language. But without more context, it's difficult to say for certain what "LLM" refers to in this specific case.

⏱️  0.83s  |  6 docs retrieved
────────────────────────────────────────────────────────────

Your question: what is attention mechanism?

💡 Answer:
According to the provided context, an attention mechanism is a function that maps a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum. 

In the context of the document, self-attention is a speci

## 14 · Batch questions (optional)

In [15]:
# Edit this list with questions specific to your PDF
BATCH_QUESTIONS = [
    'What is the main contribution of this paper?',
    'What methods or techniques are described?',
    'What are the key findings or results?',
]

print('=' * 60)
print('📝 Batch Q&A')
print('=' * 60)

for bq in BATCH_QUESTIONS:
    print(f'\n❓ {bq}')
    t0  = time.time()
    res = graph_builder.run(bq)
    print(f'💡 {res["answer"]}')
    print(f'   ⏱️  {time.time()-t0:.2f}s')
    print('-' * 60)

📝 Batch Q&A

❓ What is the main contribution of this paper?
💡 The context provided does not explicitly state the main contribution of the paper. However, based on the information given, it appears to be related to a new attention mechanism called "Scaled Dot-Product Attention" and its application in a model that achieves a new state-of-the-art BLEU score of 28.4. The paper likely presents a novel approach to attention in neural networks, but without more context, it's difficult to provide a more specific answer. 

From general knowledge, the paper "Attention Is All You Need" by Ashish Vaswani et al. is a well-known paper that introduced the Transformer model, which relies heavily on self-attention mechanisms. The main contribution of this paper is the proposal of the Transformer model, which replaces traditional recurrent neural network (RNN) and convolutional neural network (CNN) architectures with self-attention mechanisms, achieving state-of-the-art results in machine translation ta